# Model Performance Metrics Visualization
Load a JSON metrics file and produce:
1. Per-class bar plots (precision, recall, F1, IoU)
2. A colored confusion matrix

Works for both binary and multiclass models.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path

In [ ]:
# ---- CONFIGURATION ----
# Point this to your metrics JSON
METRICS_PATH = "multiclass_metrics.json"
# METRICS_PATH = "binary_metrics.json"

with open(METRICS_PATH) as f:
    metrics = json.load(f)

class_names = list(metrics["per_class"].keys())
n_classes = len(class_names)
print(f"Model type: {'binary' if n_classes == 2 else 'multiclass'} ({n_classes} classes)")
print(f"Classes: {class_names}")

## Per-Class Metrics Bar Plot

In [ ]:
metric_names = ["precision", "recall", "f1", "iou"]
metric_labels = ["Precision", "Recall", "F1", "IoU"]

# Build array: rows = classes, cols = metrics
values = np.array([
    [metrics["per_class"][c][m] for m in metric_names]
    for c in class_names
])

x = np.arange(len(metric_names))
width = 0.8 / n_classes
colors = plt.cm.Set2(np.linspace(0, 1, max(n_classes, 3)))

fig, ax = plt.subplots(figsize=(8, 5))
for i, cname in enumerate(class_names):
    offset = (i - (n_classes - 1) / 2) * width
    bars = ax.bar(x + offset, values[i], width, label=cname, color=colors[i])
    for bar, val in zip(bars, values[i]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{val:.2f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(metric_labels)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Score")
ax.set_title("Per-Class Metrics")
ax.legend(loc="upper left", framealpha=0.9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("per_class_metrics.png", dpi=200, bbox_inches="tight")
plt.show()

## Confusion Matrix

In [ ]:
cm = np.array(metrics["confusion_matrix"])

# Row-normalized for coloring (each row sums to 1)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5 + n_classes * 0.8, 4 + n_classes * 0.8))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)

# Annotate cells with raw count + percentage
for i in range(n_classes):
    for j in range(n_classes):
        pct = cm_norm[i, j]
        color = "white" if pct > 0.55 else "black"
        ax.text(j, i, f"{cm[i, j]:,}\n({pct:.1%})",
                ha="center", va="center", fontsize=9, color=color)

ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_names)
ax.set_yticklabels(class_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion Matrix (row-normalized coloring)")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Recall per class")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

## Summary Table

In [ ]:
print(f"Overall Accuracy:  {metrics['overall_accuracy']:.4f}")
print(f"Macro Precision:   {metrics['macro_precision']:.4f}")
print(f"Macro Recall:      {metrics['macro_recall']:.4f}")
print(f"Macro F1:          {metrics['macro_f1']:.4f}")
print(f"Mean IoU:          {metrics['mean_iou']:.4f}")